# OS & Shell Tutor: Training Notebook (Qwen3 + Unsloth + LoRA)

**Before you start**
1. `Runtime -> Change runtime type -> T4 GPU`
2. In Google Drive create a folder `os_tutor` and upload your dataset (`synthetic.json`) into it.
3. Run the cells **top to bottom** (Shift+Enter). Edit only the **Config** cell.

Your data must be in chat format:
```json
{"messages": [
  {"role": "system", "content": "You are an OS and shell tutor."},
  {"role": "user", "content": "What is a deadlock?"},
  {"role": "assistant", "content": "A deadlock is..."}
]}
```

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install libraries
If Colab asks you to restart the session afterwards, do it, then continue from the next cell.

In [ ]:
!pip install -q unsloth

## 3. Mount Google Drive
So your data and trained model are saved permanently (Colab's local disk is wiped when the session ends).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Config (edit this cell only)

In [ ]:
BASE_MODEL   = "unsloth/Qwen3-4B"        # try "unsloth/Qwen3-1.7B" if you run out of memory
MAX_SEQ_LEN  = 2048

# Add more files here later, e.g. ["synthetic.json", "manual_verified.json"]
DATA_FILES   = ["/content/drive/MyDrive/os_tutor/synthetic.json"]

# Change the version name for each training run: v1, v2, ...
SAVE_DIR     = "/content/drive/MyDrive/os_tutor/os_tutor_v1"

EPOCHS       = 2        # more than 3 on small synthetic data usually overfits
LEARNING_RATE = 2e-4

## 5. Load, clean and split the dataset
- **Deduplicate** on the user question so the model doesn't memorize repeats
- **Shuffle** so it doesn't learn topic order
- **5% validation split**: data the model never trains on. If validation loss rises while training loss falls, you're overfitting.

In [ ]:
import json, random
from datasets import Dataset

def load_file(path):
    with open(path) as f:
        text = f.read().strip()
    if text.startswith("["):                 # normal JSON list
        return json.loads(text)
    return [json.loads(line) for line in text.splitlines() if line.strip()]   # JSONL

data = []
for p in DATA_FILES:
    items = load_file(p)
    print(p, "->", len(items), "examples")
    data += items

print("\nFirst example:\n", json.dumps(data[0], indent=2)[:800])

In [ ]:
def user_question(ex):
    return next(m["content"] for m in ex["messages"] if m["role"] == "user")

seen, clean = set(), []
for ex in data:
    key = user_question(ex).strip().lower()
    if key not in seen:
        seen.add(key)
        clean.append(ex)

# keep only the "messages" field so the dataset is uniform
clean = [{"messages": ex["messages"]} for ex in clean]

random.seed(42)
random.shuffle(clean)

ds = Dataset.from_list(clean).train_test_split(test_size=0.05, seed=42)
train_ds, val_ds = ds["train"], ds["test"]
print(f"{len(data)} loaded -> {len(clean)} after dedup")
print(f"{len(train_ds)} train / {len(val_ds)} validation")

**Using a different format?** If your file has `instruction` / `output` instead of `messages`, run this converter *before* the dedup cell (uncomment it):

In [ ]:
# data = [{"messages": [
#     {"role": "system", "content": "You are an OS and shell tutor."},
#     {"role": "user", "content": ex["instruction"]},
#     {"role": "assistant", "content": ex["output"]},
# ]} for ex in data]

## 6. Load the base model in 4-bit
- `load_in_4bit=True` compresses weights so the model fits the T4's 16 GB (this is QLoRA).
- `max_seq_length` is the longest example (in tokens) the model handles.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

## 7. Attach LoRA adapters
LoRA freezes the base model and trains small add-on matrices instead of all the weights.
- `r`: adapter size (higher = more capacity, more memory)
- `lora_alpha`: scaling factor, usually equal to `r`
- `target_modules`: which layers get adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 8. Apply the chat template
Converts message lists into the exact text format Qwen3 expects (with tokens like `<|im_start|>`).
**Use the same template at inference time**; a mismatch is a classic cause of a "bad" fine-tune.

In [ ]:
def format_example(ex):
    text = tokenizer.apply_chat_template(
        ex["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": text}

train_ds = train_ds.map(format_example)
val_ds = val_ds.map(format_example)
print(train_ds[0]["text"])

## 9. Set up the trainer
- Batch size 2 x gradient accumulation 4 = effective batch of 8
- `train_on_responses_only` makes the model learn from the **assistant answers only**, not the questions
- Validation loss is printed every 50 steps

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,      # on older trl versions use: tokenizer=tokenizer
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_steps=5,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

## 10. Train
Healthy training: both training and validation loss go **down**. If validation loss starts rising while training loss keeps falling, use fewer epochs.

In [ ]:
trainer.train()

## 11. Quick test

In [ ]:
FastLanguageModel.for_inference(model)   # switch to fast generation mode

def ask(question, system="You are an OS and shell tutor.", max_new_tokens=512):
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         temperature=0.7, top_p=0.8, top_k=20, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask("Explain the difference between a process and a thread."))

## 12. Save the LoRA adapter to Drive
The adapter is small (usually under 200 MB). **This is what the chatbot notebook loads.**

In [ ]:
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to", SAVE_DIR)

## 13. (Optional) Export a merged model or GGUF
Only needed if you want to use the model outside Unsloth (Transformers, vLLM, LM Studio, llama.cpp). GGUF export builds llama.cpp in the background and can take several minutes.

In [ ]:
# Merged 16-bit model (adapter baked into base weights)
# model.save_pretrained_merged(SAVE_DIR + "_merged", tokenizer, save_method="merged_16bit")

# GGUF for LM Studio / llama.cpp
# model.save_pretrained_gguf(SAVE_DIR + "_gguf", tokenizer, quantization_method="q4_k_m")

## Next steps
- **Add more data:** put new files in `DATA_FILES` (Config cell), change `SAVE_DIR` to `..._v2`, and use **Runtime -> Restart session and run all**. Retraining from the base model on old + new data is the safest approach.
- Open the **`os_tutor_rag_chatbot`** notebook to add RAG and the chat interface.